# Detecting silent 200s in the scrape pipeline

**Status:** experimental, behind the flag `__experimental_catch_silent_failures` (default off). The detector itself is one commit — [implementation diff](https://github.com/xbt-a4224j/firecrawl/commit/a1c4cf4568b5b8b7c1eb5a84f548f2d64d00623c) — and the full branch (detector + this walkthrough) is [`main...feat/scrape-quality-signal`](https://github.com/xbt-a4224j/firecrawl/compare/main...feat/scrape-quality-signal).

## Summary

`/scrape` treats any non-empty response body as a success: the engine acceptance gate is
`markdown.length > 0`. So a scrape can return `success: true` with HTTP 200 while the content is a
bot wall, a paywall or login/consent gate, or an error/placeholder page. We call this a *silent 200*.

![what the flag changes (flag off vs on)](value.svg)

This notebook is a reproduction and evaluation: (1) demonstrate the failure, (2) describe a
content-signature detector, (3) evaluate it against two labeled corpora and report precision and
recall. Everything runs against the real API.

## 0 — Environment

`up.sh` brings up the stack and is idempotent: a no-op if the API is already healthy, otherwise
`docker compose up` (Playwright runs in a Linux container). Readiness is confirmed by a real scrape,
not an open port.

In [1]:
// stand up the stack (idempotent — no-op if already healthy) and define our one helper.
const _up = await new Deno.Command("bash", { args: ["up.sh"] }).output();
console.log(new TextDecoder().decode(_up.stdout).trim());
if (_up.code !== 0) console.error(new TextDecoder().decode(_up.stderr).trim());

const API = Deno.env.get("FIRECRAWL_API") ?? "http://localhost:3002";

type Scrape = { success: boolean; status: number | null; markdown: string; error: string | null };
async function scrape(url: string, catchSilent = false): Promise<Scrape> {
  try {
    const r = await fetch(`${API}/v2/scrape`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ url, formats: ["markdown"], __experimental_catch_silent_failures: catchSilent, timeout: 45000 }),
      signal: AbortSignal.timeout(60000),
    });
    const j = await r.json();
    return { success: j.success === true, status: j.data?.metadata?.statusCode ?? null, markdown: j.data?.markdown ?? "", error: j.error ?? null };
  } catch (e) {
    return { success: false, status: null, markdown: "", error: String((e as Error)?.message ?? e) };
  }
}
console.log(`\nhelper ready -> scrape(url, catchSilent?) against ${API}`);

▶ checking http://localhost:3002 …
✓ stack already healthy — nothing to do

helper ready -> scrape(url, catchSilent?) against http://localhost:3002


## 1 — Problem: HTTP status is not a content-quality signal

A single request illustrates the gap. The pipeline reports the scrape as a success, but the returned
content is an anti-bot challenge page. `success` is `true`; the body is unusable.

This is a documented, recurring failure. The clearest example is
[#1363](https://github.com/firecrawl/firecrawl/issues/1363): a self-hosted scrape returns `success: true` with markdown reading
*"Verify you are human by completing the action below…"* and `statusCode: 403` in the metadata — a
Cloudflare interstitial accepted as content. Related: [#495](https://github.com/firecrawl/firecrawl/issues/495) (blocked by Cloudflare),
[#2257](https://github.com/firecrawl/firecrawl/issues/2257) (anti-bot: the browser engine returns the block page rather than failing over),
and [#1309](https://github.com/firecrawl/firecrawl/issues/1309) (a crawl completes and returns an empty result with no error).

In [2]:
// One junk URL, default behavior. The scraper declares success; the "content" is a captcha page.
const r = await scrape("https://www.sedarplus.ca/csa-party/party/document.html?partyType=issuer");
console.log("success:", r.success, "  HTTP:", r.status);
console.log("the 'content' it returned:");
console.log(" ", r.markdown.slice(0, 160).replace(/\s+/g, " "));

success: true   HTTP: 200
the 'content' it returned:
  ![Captcha Page](https://captcha.perfdrive.com/captcha-public/images/ss_captcha.png) We apologize for the inconvenience... =====================================


## 2 — Approach: signatures on the extracted markdown

First, where this sits in the full scrape lifecycle. The red **SILENT 200 ZONE** is the gap: a 200 with no content check anywhere in the pipeline.

![full scrape lifecycle](pipeline.svg)

The detector is four pure functions over the already-extracted markdown — bot wall, soft error,
gate, placeholder — each matching only the *lead* of the content. The lead constraint is the
precision mechanism: a real page that embeds a captcha widget or a cookie banner still opens with
content, whereas a wall opens with the challenge text. A rejected result re-enters the engine
fallback loop, so the pipeline escalates rather than simply failing.

![acceptance gate: where the flag inserts](gate.svg)

![zoom: the accept/reject decision](zoom.svg)

A length-based "thin content" heuristic was considered and rejected. The rationale is in the source
note printed below, and is quantified in section 4.

The change to the pipeline is small — the acceptance gate gains one term (`scrapeURL/index.ts`):

```diff
+ // flag-gated: run the four heuristics on the already-extracted markdown
+ if (meta.options.__experimental_catch_silent_failures) {
+   silentFailures = detectSilentFailure(checkMarkdown);
+ }
  ...
+ const isSilentFailure = silentFailures.length > 0;
- if (isLongEnough || !isGoodStatusCode) {
+ if ((isLongEnough && !isSilentFailure) || !isGoodStatusCode) {
    return engineResult;                         // accepted
  } else {
    throw new EngineUnsuccessfulError(engine);   // rejected -> next engine in the fallback waterfall
  }
```

In [3]:
// The detector (snapshot of apps/api/.../scrapeURL/lib/silentFailure.ts). The honest part is what is NOT there:
const src = await Deno.readTextFile("./silentFailure.ts");   // snapshot copied alongside the notebook
const srcLines = src.split("\n");
console.log(srcLines.filter((l, i) => i >= 29 && i <= 35).join("\n"));   // why no length heuristic
console.log("");
for (const l of srcLines) if (/^\/\/ #\d /.test(l)) console.log(l.replace(/ Deliberately.*/, "").trim());

// NOTE: a length-based "thin content" heuristic was deliberately rejected. Measured against a
// known-good corpus, real short pages and thin junk overlap completely — example.com (134 readable
// chars) is the same length as the excalidraw JS shell (134) and barely shorter than a real bot
// wall (153). No length threshold separates them, so any length rejector breaks example.com. The
// truly-empty case is already handled upstream by the engine gate's `length > 0` check. So we only
// reject on SPECIFIC signatures below — precision over recall.


// #1 BOT_WALL — challenge copy in the lead, or a dedicated anti-bot asset domain.
// #2 SOFT_ERROR — 200 with an error page the app rendered instead of content.
// #3 GATE — login/subscribe/paywall gate where the content should be.
// #4 PLACEHOLDER — skeleton "Loading…" that never populated. Require the ellipsis/dots so a real


## 3 — Evaluation: labeled corpora

Two corpora: `TRUE_POSITIVES` (known silent 200s, measuring recall) and `FALSE_POSITIVES`
(known-good pages, several chosen as near-misses, measuring precision). Each URL is scraped twice,
flag off and on. `caught` denotes a result the flag turned from success into failure.

Precision is treated as a hard constraint: a content gate that rejects a valid page is a regression.
Recall is reported as measured.

In [4]:
// Two labeled corpora — a representative slice of the full e2e dataset.
const TRUE_POSITIVES: [string, string][] = [
  ["https://www.sedarplus.ca/csa-party/party/document.html?partyType=issuer", "bot wall (PerimeterX)"],
  ["https://opencorporates.com/companies/gb/00000006", "bot wall (Cloudflare)"],
  ["https://www.bizapedia.com/companies/apple-inc.html", "bot wall (reCAPTCHA)"],
  ["https://www.academia.edu/12345678/Test_Paper", "signup gate"],
  ["https://www.quora.com/profile/Adam-DAngelo", "soft error"],
  ["https://www.tiktok.com/@nasa", "JS shell (no signature)"],
  ["https://vscode.dev", "JS shell (no signature)"],
];
const FALSE_POSITIVES: [string, string][] = [
  ["https://example.com", "tiny real page"],
  ["https://en.wikipedia.org/wiki/Cloudflare", "article ABOUT Cloudflare"],
  ["https://en.wikipedia.org/wiki/CAPTCHA", "article ABOUT CAPTCHAs"],
  ["https://www.lusha.com/company-search/apple/0e3f1a/", "embeds a reCAPTCHA widget"],
  ["https://www.statista.com/statistics/272014/global-social-networks-ranked-by-number-of-users/", "real chart + consent banner"],
  ["https://www.w3.org/TR/html52/", "says 'loading' in prose"],
  ["https://jestjs.io", "docs"],
  ["https://www.python.org", "homepage"],
];

async function pool<T, R>(items: T[], n: number, fn: (t: T) => Promise<R>): Promise<R[]> {
  const out = new Array<R>(items.length); let i = 0;
  await Promise.all(Array.from({ length: Math.min(n, items.length) }, async () => {
    while (i < items.length) { const j = i++; out[j] = await fn(items[j]); }
  }));
  return out;
}
const run = (rows: [string, string][]) => pool(rows, 6, async ([url, note]) => {
  const off = await scrape(url, false);
  const on = await scrape(url, true);
  return { site: url.replace(/^https?:\/\//, "").slice(0, 38), note, off: off.success, on: on.success };
});

const tp = await run(TRUE_POSITIVES);
const fp = await run(FALSE_POSITIVES);
const caught = tp.filter((r) => r.off && !r.on).length;
const regressions = fp.filter((r) => r.off && !r.on).length;

const pad = (s: string, n: number) => (s.length > n ? s.slice(0, n - 1) + "…" : s).padEnd(n);
const cols = (a: string, b: string, c: string, d: string, e: string) => "  " + [pad(a, 30), pad(b, 27), pad(c, 7), pad(d, 7), e].join("  ");
const showTable = (header: string, rows: { site: string; note: string; off: boolean; on: boolean }[], verdict: (off: boolean, on: boolean) => string) => {
  console.log(header);
  console.log(cols("site", "kind", "off", "on", "result"));
  console.log("  " + "-".repeat(64));
  for (const r of rows) console.log(cols(r.site, r.note, r.off ? "success" : "fail", r.on ? "success" : "fail", verdict(r.off, r.on)));
};

showTable("TRUE_POSITIVES — known silent 200s (measures recall)", tp, (off, on) => (off && !on ? "caught" : "missed"));
console.log("");
showTable("FALSE_POSITIVES — known-good pages (measures precision)", fp, (off, on) => (off && !on ? "REGRESSION" : "kept"));
console.log(`\nRECALL  ${caught}/${tp.length} junk caught   ·   PRECISION  ${fp.length - regressions}/${fp.length} good pages kept`);

TRUE_POSITIVES — known silent 200s (measures recall)
  site                            kind                         off      on       result
  ----------------------------------------------------------------
  www.sedarplus.ca/csa-party/pa…  bot wall (PerimeterX)        success  fail     caught
  opencorporates.com/companies/…  bot wall (Cloudflare)        success  fail     caught
  www.bizapedia.com/companies/a…  bot wall (reCAPTCHA)         success  fail     caught
  www.academia.edu/12345678/Tes…  signup gate                  success  fail     caught
  www.quora.com/profile/Adam-DA…  soft error                   success  fail     caught
  www.tiktok.com/@nasa            JS shell (no signature)      success  success  missed
  vscode.dev                      JS shell (no signature)      success  success  missed

FALSE_POSITIVES — known-good pages (measures precision)
  site                            kind                         off      on       result
  -----------------------------

## 4 — Results and limitations

- **Precision.** No good page is rejected, including the near-misses: an article *about* Cloudflare,
  a page embedding a reCAPTCHA widget, and articles served behind consent banners.
- **Recall.** The flag catches silent 200s that carry a specific signature and misses those that do
  not (e.g. a bare JS shell). The reported recall reflects this; it is not 1.0.
- **On the length heuristic.** A `THIN_CONTENT` rule by markdown length was the obvious first
  approach. The false-positive corpus rejected it — the readable lengths of valid and junk pages
  interleave, so no threshold separates them:

![readable length: real and junk pages overlap](overlap.svg)

- **Mechanism.** Detection runs inside the engine fallback loop. A rejected result is treated like an
  engine failure, so the pipeline escalates to the next engine before returning an error.

The flag is off by default; with it off, the acceptance path is unchanged.